# 13. Baseline 학습과 공통 평가지표 구현

이 노트북은 Hugging Face의 실제 SegFormer-B0 체크포인트를 SyntheticMetalSeg train split에 fine-tuning하고, hold-out split 전체에 대해 실제 추론을 수행합니다.

저장되는 `sample_metrics.csv`, `group_metrics.csv`, `class_metrics.csv`가 14-21번 노트북의 근거 데이터입니다.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "ch2_utils.py").exists():
    matches = (
        list(Path.cwd().glob("Deeplearning/*/2-1장/ch2_utils.py"))
        + list(Path.cwd().glob("Deeplearning/*/2장/ch2_utils.py"))
        + list(Path.cwd().glob("**/ch2_utils.py"))
    )
    NOTEBOOK_DIR = matches[0].parent if matches else Path("Deeplearning") / "Vision 응용" / "2-1장"
sys.path.append(str(NOTEBOOK_DIR))

from ch2_utils import *

paths = find_paths()
set_korean_font()
set_seed(7)
samples = load_samples(paths.data_root)
paths

## 13-1. 학습/평가 manifest 준비

In [ ]:
train_manifest, eval_manifest = create_standard_manifests(samples, paths.runs_root)
run_dir = paths.runs_root / "baseline_segformer_b0"
print("model:", SEGFORMER_B0_MODEL_NAME)
print(train_manifest)
print(eval_manifest)

## 13-2. SegFormer-B0 학습과 추론

In [ ]:
EPOCHS = 3
BATCH_SIZE = 8
LR = 1e-3

if not (run_dir / "sample_metrics.csv").exists():
    result = train_segformer_experiment(
        train_manifest=train_manifest,
        eval_manifest=eval_manifest,
        run_dir=run_dir,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LR,
        seed=7,
        augment=False,
        model_name=SEGFORMER_B0_MODEL_NAME,
        use_pretrained=True,
    )
else:
    print("이미 baseline 추론 결과가 있어 재학습을 건너뜁니다:", run_dir)

sample_metrics, group_metrics, class_metrics = load_run_metrics(run_dir)
display(class_metrics)
display(group_metrics.head(20))

## 13-3. 실제 추론 결과 예시

In [ ]:
import importlib
import ch2_utils as ch2u

ch2u = importlib.reload(ch2u)
if (run_dir / "checkpoint.pt").exists():
    example_path = ch2u.save_prediction_examples(eval_manifest, run_dir, n=6, seed=7)
    print(example_path)
    display(ch2u.Image.open(example_path))
else:
    print("checkpoint is missing. Run cell 13-2 first:", run_dir / "checkpoint.pt")

## 13-4. 공통 지표 결론

In [ ]:
print(conclusion_from_group(group_metrics, "color_group", "target_dice_mean"))
print(conclusion_from_group(group_metrics, "defect_type", "target_dice_mean"))
print(conclusion_from_group(group_metrics, "shape_group", "target_dice_mean"))